<a href="https://colab.research.google.com/github/Marconiadsf/fiap-postech-datathon/blob/main/Fase_5_Datathon_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0. Configuração

In [8]:
#@title Concentrar a importacão de bibliotecas nesta célula

import pandas as pd
import numpy as np

In [9]:
#@title Concentrar o caregamento de dados nesta célula
# Url configurada para o repo do curso:
url_planilha = "https://docs.google.com/spreadsheets/d/1td91KoeSgXrUrCVOUkLmONG9Go3LVcXpcNEw_XrL2R0/export?format=xlsx"

try:
    planilhas_excel = pd.read_excel(url_planilha, sheet_name=None)
    print("Planilha carregada via URL com sucesso!")
except Exception as erro_execucao:
    print(f"Erro ao acessar o link fornecido. Erro: {erro_execucao}")
    print("Verifique se a planilha ainda existe no endereço informado e \
     se tem permissão de 'Qualquer pessoa com o link' para leitura.")


Planilha carregada via URL com sucesso!


# 1. Tratamento inicial de arquivos

In [10]:
#@title Tratamento inicial dos dados

planilhas_excel.keys()


dict_keys(['PEDE2022', 'PEDE2023', 'PEDE2024'])

In [11]:
df = planilhas_excel
colunas_unicas = sorted(set([*df["PEDE2022"], *df["PEDE2023"], *df["PEDE2024"]]), key=lambda x: x.lower())
print("Colunas únicas: \n", colunas_unicas)
for df_name, df in planilhas_excel.items():
  print(f"Colunas do dataframe {df_name} : ", sorted([*df], key=lambda x: x.lower()))

Colunas únicas: 
 ['Ano ingresso', 'Ano nasc', 'Atingiu PV', 'Ativo/ Inativo', 'Ativo/ Inativo.1', 'Avaliador1', 'Avaliador2', 'Avaliador3', 'Avaliador4', 'Avaliador5', 'Avaliador6', 'Cf', 'Cg', 'Ct', 'Data de Nasc', 'Defas', 'Defasagem', 'Destaque IDA', 'Destaque IEG', 'Destaque IPV', 'Destaque IPV.1', 'Escola', 'Fase', 'Fase ideal', 'Fase Ideal', 'Gênero', 'IAA', 'IAN', 'IDA', 'Idade', 'Idade 22', 'IEG', 'INDE 2023', 'INDE 2024', 'INDE 22', 'INDE 23', 'Indicado', 'Ing', 'Inglês', 'Instituição de ensino', 'IPP', 'IPS', 'IPV', 'Mat', 'Matem', 'Nome', 'Nome Anonimizado', 'Nº Av', 'Pedra 20', 'Pedra 2023', 'Pedra 2024', 'Pedra 21', 'Pedra 22', 'Pedra 23', 'Por', 'Portug', 'RA', 'Rec Av1', 'Rec Av2', 'Rec Av3', 'Rec Av4', 'Rec Psicologia', 'Turma']
Colunas do dataframe PEDE2022 :  ['Ano ingresso', 'Ano nasc', 'Atingiu PV', 'Avaliador1', 'Avaliador2', 'Avaliador3', 'Avaliador4', 'Cf', 'Cg', 'Ct', 'Defas', 'Destaque IDA', 'Destaque IEG', 'Destaque IPV', 'Fase', 'Fase ideal', 'Gênero', 'IAA'

Problemas:

1. O nome de Aluno Anonimizado não entrega informação (RA já tem o número).
2. Colunas de idade relativa podem ser removidas ou renomeadas
3. Colunas são variações de nome ou duplicatas:
 - Ano nasc, Data de Nasc
 - Ativo/Inativo, Ativo/Inativo.1
 - Defas, Defasagem?
 - Destaque IPV, Destaque IPV.1?
 - Fase Ideal, Fase ideal
 - INDE 2023, INDE 23
 - Ing,Ingles?
 - Mat,Matem
 - Pedra 2023,Pedra 23
 - Por,Portug
 - Idade 22, Idade 22
Observando os dados visualmente conclui-se que podemos renomear as colunas e fazer fusão das colunas

In [12]:
#@title Consolidação dos dados

def consolidar_colunas_repetidas(dataframe):
    """
    Percorre o dataframe em busca de colunas com nomes idênticos e as funde
    seguindo a regra: prioriza valores não nulos e valida consistência.
    """
    lista_colunas = pd.Series(dataframe.columns)
    nomes_duplicados = lista_colunas[lista_colunas.duplicated()].unique()

    for nome_coluna in nomes_duplicados:
        # Seleciona apenas as colunas que possuem o nome repetido
        subconjunto_duplicado = dataframe.loc[:, dataframe.columns == nome_coluna]

        def unir_linhas_conflitantes(linha):
            # Remove valores nulos (NaN) para análise
            valores_existentes = linha.dropna().unique()

            if len(valores_existentes) == 0:
                return np.nan

            if len(valores_existentes) == 1:
                return valores_existentes[0]

            # Se houver mais de um valor único diferente, dispara erro de consistência
            erro_mensagem = f"Conflito de dados na coluna '{nome_coluna}': valores divergentes encontrados {valores_existentes}"
            raise ValueError(erro_mensagem)

        # Aplica a lógica de união linha a linha
        coluna_unificada = subconjunto_duplicado.apply(unir_linhas_conflitantes, axis=1)

        # Remove as colunas duplicadas antigas e insere a nova versão consolidada
        dataframe = dataframe.loc[:, dataframe.columns != nome_coluna].copy()
        dataframe[nome_coluna] = coluna_unificada

    return dataframe

mapeamento_padronizacao = {
    'Ano nasc': 'Ano Nasc',
    'Data de Nasc': 'Ano Nasc',
    'Defas': 'Defasagem',
    'Destaque IPV.1': 'Destaque IPV',
    'Fase ideal': 'Fase Ideal',
    'INDE 23': 'INDE 2023',
    'Ingles': 'Ing',
    'Inglês': 'Ing',
    'Matem': 'Mat',
    'Pedra 23': 'Pedra 2023',
    'Portug': 'Por',
    'Idade 22': 'Idade',
}

for nome_aba, aba in planilhas_excel.items():
    # 1. Identificação do ano extraída do nome da aba
    ano_extraido = int(''.join(filter(str.isdigit, nome_aba)))
    aba['ANO'] = ano_extraido

    # 2. Padronização dos nomes conforme dicionário de mapeamento
    aba = aba.rename(columns=mapeamento_padronizacao)

    # 3. Execução da limpeza de duplicatas internas
    planilhas_excel[nome_aba] = consolidar_colunas_repetidas(aba)

# União final de todas as abas tratadas em um único DataFrame
df_concatenado = pd.concat(planilhas_excel.values(), ignore_index=True)

# Removendo a coluna Nome Anonimizado, Nome, e 'Ativo/ Inativo.1'
# Por que? : Nomes não estão agregando valor, 'Ativo/ Inativo.1'
# é duplicata de 'Ativo/ Inativo'
df_limpo = df_concatenado.copy()
df_limpo.drop(columns=['Nome Anonimizado', 'Nome', 'Ativo/ Inativo.1'], inplace=True)
print(sorted([*df_limpo], key=lambda x: x.lower()))



['ANO', 'Ano ingresso', 'Ano Nasc', 'Atingiu PV', 'Ativo/ Inativo', 'Avaliador1', 'Avaliador2', 'Avaliador3', 'Avaliador4', 'Avaliador5', 'Avaliador6', 'Cf', 'Cg', 'Ct', 'Defasagem', 'Destaque IDA', 'Destaque IEG', 'Destaque IPV', 'Escola', 'Fase', 'Fase Ideal', 'Gênero', 'IAA', 'IAN', 'IDA', 'Idade', 'IEG', 'INDE 2023', 'INDE 2024', 'INDE 22', 'Indicado', 'Ing', 'Instituição de ensino', 'IPP', 'IPS', 'IPV', 'Mat', 'Nº Av', 'Pedra 20', 'Pedra 2023', 'Pedra 2024', 'Pedra 21', 'Pedra 22', 'Por', 'RA', 'Rec Av1', 'Rec Av2', 'Rec Av3', 'Rec Av4', 'Rec Psicologia', 'Turma']


In [13]:
df_limpo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3030 entries, 0 to 3029
Data columns (total 51 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   RA                     3030 non-null   object 
 1   Fase                   3030 non-null   object 
 2   Turma                  3030 non-null   object 
 3   Ano Nasc               3030 non-null   object 
 4   Idade                  3030 non-null   object 
 5   Gênero                 3030 non-null   object 
 6   Ano ingresso           3030 non-null   int64  
 7   Instituição de ensino  3029 non-null   object 
 8   Pedra 20               754 non-null    object 
 9   Pedra 21               1061 non-null   object 
 10  Pedra 22               1932 non-null   object 
 11  INDE 22                1932 non-null   float64
 12  Cg                     860 non-null    float64
 13  Cf                     860 non-null    float64
 14  Ct                     860 non-null    float64
 15  Nº A

**Observação sobre tratamento de dados: A essa altura os dados estão consolidados no dataframe porém o conteúdo, i.e. o dado em si não está tratado. Conforme a evolução do trabalho outros tratamentos podem ser adicionados, por
ém faz mais sentido deixar os dados brutos no momento para serem trabalhados sob cada pergunta.**

#2. Perguntas e respostas
